# MIS584 Lab Assignment 2
## Learning Objectives
* Demonstrate the use of PySpark RDD for basic distributed computing
* Demonstrate the use of PySpark SQL for structured data analysis

## Due Date
**Check D2L for the Due Dates**

## Assignment Submission Instructions
When your file is ready, submit the following deliverables to the Lab Assignmen 2 dropbox:
* Provide the link to your Google Colab notebook in the comments section; please make sure that **you enable the general access to your notebook with links before submission**. Failure to open your notebook will automatically lead to a grade of 0.
* Upload the notebook file with the `.ipynb` suffix to the submission drop box. The uploaded notebook should have the same content as the one shared through the link, include enough documentation of the code, and have all the outputs available.

## Others
As always, feel free to come to our office hours or let us know through email if you face any difficulties/challenges while finishing the assignment. Good luck! For your convenience, I have created the text and code cells you might need for the lab assignment. Please also complete your contact information in the notebook as well.

## Student's Contact Information:
Name:Ripa Shah

Email: ripashah@arizona.edu

## Part 0: Configure Spark and Download Data
In this section, we provide you with the code to install PySpark, configure Spark Context and Spark Session, and download the datasets you are going to use. You can safely run the code in this section. After everything finishes successfully, you will see two datasets, one named `text_reviews.txt` and the other named `user_reviews.csv` located under your Google Colab workspace. Alternatively, you can copy and paste the url links to both datasets in your web browser and download them to your local desktop as well.

In [1]:
%pwd

'c:\\Users\\yashs\\source\\repos\\Ripa-Shah\\BigDataTechnology\\Big-Data-Technology'

In [2]:
!pip install pyspark

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\yashs\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# create spark context and spark session
from pyspark import SparkConf, SparkContext



In [4]:
%pwd

'c:\\Users\\yashs\\source\\repos\\Ripa-Shah\\BigDataTechnology\\Big-Data-Technology'

In [5]:
# Source - https://stackoverflow.com/a
# Posted by Sanjeev v, modified by community. See post 'Timeline' for change history
# Retrieved 2025-11-08, License - CC BY-SA 4.0

import os       #importing os to set environment variable
def install_java():
  !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null      #install openjdk
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"     #set environment variable
  !java -version       #check java version
install_java()


The system cannot find the path specified.
'java' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
conf = SparkConf().setMaster("local").setAppName("MIS584_Lab_Assignment2")




In [ ]:
sc = SparkContext(conf = conf)

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MIS584_Lab_Assignment2").getOrCreate()

In [ ]:
# download the yelp review dataset
from urllib.request import urlretrieve
urlretrieve('https://drive.google.com/uc?export=download&id=1AV5z7EcHoMabL5yijp_jRwcqzbHNhxGp',
            'user_reviews.csv') ## Save url's file locally.
urlretrieve('https://drive.google.com/uc?export=download&id=1D1BKYITdAsy82UVcbO8bTL7oDY-flHId',
            'text_reviews.txt')

('text_reviews.txt', <http.client.HTTPMessage at 0x794522168050>)

## Part 1: Practice of PySpark RDD (4 points)
In this part, you need to use the techniques for PySpark RDD to read and process an external file that stores users' text reviews on businesses from Tucson.

Specifically, the external file, `text_reviews.txt`, has two columns separated by tabs (i.e., `\t`). The first column stores each review's associated userr's ID, and the second column stores each review's text inforamtion. Please use the Spark techniques we discussed during the class to finish the following tasks:
1. Read the text file using Spark's `textFile` function into an RDD; extract each line's user ID and review text by creating a customized mapper function and call the `map` transformation on the RDD. (1 point)
2. Use the `distinct` action to return the total number of unique users in the dataset. Print the number. (0.5 point)
3. Get the total number of reviews each user has posted using the `reduceByKey` transformation. Sort the transformed RDD by values (i.e., total number of reviews) in the descending order with the `sortBy` transformation. Print top 10 users who post the largest amount of reviews. (0.75 point)
4. Get the total length of reviews (i.e., the total number of characters) each user has posted using the `reduceByKey` transformation. Sort the transformed RDD by values (i.e., total length of reviews) in the ascending order with the `sortBy` transformation. Print the 10 most quiet users (with the lowest total length of reviews). (0.75 point)

**Hint**: For task 4, you might want to map each `(user_id, review_text)` tuple into `(user_id, len(review_text))`. After that, you can use the `reduceByKey` transformation to aggregate total length of reviews by users.



In [ ]:
import csv
from io import StringIO

In [ ]:
lines = sc.textFile("text_reviews.txt")

lines.take(5)

["CMYCfKoEu0WF9_43zRgr8g\tWe love this little restaurant! It's not as overrated  and loud compared to other places in Tucson. Sushi is awesome and for the right price.",
 "CMYCfKoEu0WF9_43zRgr8g\tWe came here for dinner this evening and was absolutely delicious! Truly I was blown away. I ordered a fish special at market price and couldn't believe how yummy it was. Can't recall the name of the fish but it was in the grouper family. It tasted very identical to crab meat.   I had my one year old daughter with and couldn't get her to eat anything, but she loved my fish! Service was friendly. Love it here, but it is pricey! $100+ for two people, however every penny was well spent.",
 "CMYCfKoEu0WF9_43zRgr8g\tJust a heads up the owner, Roya, will not give refunds or respond to clients. I paid $500+ to reserve hair and makeup for my bridal party in September.   I was not informed that the business was closed and haven't received any responses regarding to getting a refund. Roya is a total sca

In [ ]:
import csv
from io import StringIO

# Read the file
reviews_rdd = sc.textFile("text_reviews.txt")

# Define robust CSV parser
def line_parser(line):
    if not line or line.strip() == "":
        return None
    parts = line.split("\t")
    if len(parts) == 2:
      user_id = parts[0].strip()
      review_text = parts[1].strip()
    return (user_id, review_text)

# Apply map + filter
user_reviews_rdd = reviews_rdd.map(line_parser).filter(lambda x: x is not None)

# Show a few results
print("First 5 user-review pairs:")
for uid, review in user_reviews_rdd.take(5):
    print(uid + "|", review)

First 5 user-review pairs:
CMYCfKoEu0WF9_43zRgr8g| We love this little restaurant! It's not as overrated  and loud compared to other places in Tucson. Sushi is awesome and for the right price.
CMYCfKoEu0WF9_43zRgr8g| We came here for dinner this evening and was absolutely delicious! Truly I was blown away. I ordered a fish special at market price and couldn't believe how yummy it was. Can't recall the name of the fish but it was in the grouper family. It tasted very identical to crab meat.   I had my one year old daughter with and couldn't get her to eat anything, but she loved my fish! Service was friendly. Love it here, but it is pricey! $100+ for two people, however every penny was well spent.
CMYCfKoEu0WF9_43zRgr8g| Just a heads up the owner, Roya, will not give refunds or respond to clients. I paid $500+ to reserve hair and makeup for my bridal party in September.   I was not informed that the business was closed and haven't received any responses regarding to getting a refund. Ro

In [ ]:
userids = user_reviews_rdd.map(lambda x: x[0])

userids.take(5)

['CMYCfKoEu0WF9_43zRgr8g',
 'CMYCfKoEu0WF9_43zRgr8g',
 'CMYCfKoEu0WF9_43zRgr8g',
 'CMYCfKoEu0WF9_43zRgr8g',
 'CMYCfKoEu0WF9_43zRgr8g']

In [ ]:
useridRDD = userids.distinct()

useridRDD.take(5)


['CMYCfKoEu0WF9_43zRgr8g',
 'aMP9YyQbzcxXTGkWEy1kvg',
 'wl7VR72-u8ADycZAwGC8gQ',
 'iQnNxuxBEGb5NJBBRK-LVQ',
 'cBbVbcqUWrgYLP-06v0UXA']

In [ ]:
userids_count = useridRDD.count()

print("\n Count of Distinct userids:",userids_count)


 Count of Distinct userids: 200


In [ ]:
user_review_counts_rdd = user_reviews_rdd.map(lambda x: (x[0], 1))

In [ ]:
user_review_counts_rdd.take(5)

[('CMYCfKoEu0WF9_43zRgr8g', 1),
 ('CMYCfKoEu0WF9_43zRgr8g', 1),
 ('CMYCfKoEu0WF9_43zRgr8g', 1),
 ('CMYCfKoEu0WF9_43zRgr8g', 1),
 ('CMYCfKoEu0WF9_43zRgr8g', 1)]

In [ ]:
total_reviews_by_user = user_review_counts_rdd.reduceByKey(lambda a, b: a + b)
total_reviews_by_user.take(50)

[('CMYCfKoEu0WF9_43zRgr8g', 6),
 ('aMP9YyQbzcxXTGkWEy1kvg', 5),
 ('wl7VR72-u8ADycZAwGC8gQ', 12),
 ('iQnNxuxBEGb5NJBBRK-LVQ', 14),
 ('cBbVbcqUWrgYLP-06v0UXA', 5),
 ('b-hqAAJi54uABWGiy_XlQg', 7),
 ('8QhxpmDpd-_41qtQKQmI9Q', 7),
 ('fNbARukUV1ppV2h047pU8g', 7),
 ('8QzSs8W34_PMvIuvvXxKKg', 11),
 ('1brn4qMCfq1qilTWrF6I5Q', 7),
 ('_-SjDVit-tv9nl4qFtltZQ', 14),
 ('OztI_L9xtMIrj-kPzWk9Lg', 9),
 ('rejYbLHPQT9M9UjvvR4wsQ', 6),
 ('owPTW5g8soyD9c07QsdYtg', 6),
 ('mt0tysv5yI7YQpKXzly12w', 5),
 ('tmfv3LSML-vEuEey5aXAzw', 9),
 ('75ZCTSy7klh6q8qajxW1Cg', 11),
 ('uQGZjgALpl5ldlc9VvtBSg', 7),
 ('zRCCmHmS1dshRUAcL5fnMg', 8),
 ('LPovj-Wa7xXoJKM3qRy1bA', 16),
 ('KDIfvIBLCF00q8Ew1W23mw', 9),
 ('VIo2O8mZ3CHgvWvKAtb3lQ', 8),
 ('MV4o5u9FfQhdKNbsat7JhQ', 6),
 ('LdRkF_b7pvkjrmy3ZKEiig', 16),
 ('N6JTGCIayJiZtBGVl3Hsog', 7),
 ('kbmUOqtVz4rB64zUpFKtnA', 16),
 ('0yWkedf4StfpU9JRzKR3UQ', 5),
 ('PlNxOhtu67EZpsV--IYDXg', 11),
 ('FoGbcS6aWuryrvhIeTvShw', 11),
 ('E9Vs89OXrFEcQ4k46YPitA', 6),
 ('mvwL74bBrEKMmcY0ySDsKA', 6)

user_review_counts_rdd.take(5)

In [ ]:
top_reviews = total_reviews_by_user.sortBy(lambda x: x[1], ascending=False)
top_reviews.take(10)

[('r0pPV4-xj1sD_uGXVYxOaw', 20),
 ('jn_dHhsCj2scx1951CKutA', 20),
 ('Azxo0oP96tot8QGruS4XZw', 19),
 ('Gs4OijDfrHzAbocJ-YYGog', 18),
 ('uGbRVMSgnWKJN4lxLAABQw', 18),
 ('8OHkSxQRVfmMu5uQATi83g', 18),
 ('nl8HXOlCwIJ86pYavbi5UQ', 17),
 ('oiZUTKnsIilXwyN-HCK55w', 17),
 ('LPovj-Wa7xXoJKM3qRy1bA', 16),
 ('LdRkF_b7pvkjrmy3ZKEiig', 16)]

In [ ]:
user_Review_length_RDD = user_reviews_rdd.map(lambda x: (x[0], len(x[1])))

user_Review_length_RDD.take(5)

[('CMYCfKoEu0WF9_43zRgr8g', 141),
 ('CMYCfKoEu0WF9_43zRgr8g', 488),
 ('CMYCfKoEu0WF9_43zRgr8g', 637),
 ('CMYCfKoEu0WF9_43zRgr8g', 507),
 ('CMYCfKoEu0WF9_43zRgr8g', 322)]

In [ ]:
user_review_counts_sort_rdd = user_Review_length_RDD.reduceByKey(lambda a, b: a + b)

user_review_counts_sort_rdd.take(5)

[('CMYCfKoEu0WF9_43zRgr8g', 2196),
 ('aMP9YyQbzcxXTGkWEy1kvg', 1360),
 ('wl7VR72-u8ADycZAwGC8gQ', 12104),
 ('iQnNxuxBEGb5NJBBRK-LVQ', 3748),
 ('cBbVbcqUWrgYLP-06v0UXA', 6999)]

In [ ]:
top_reviews = user_review_counts_sort_rdd.sortBy(lambda x: x[1], ascending=False)
top_reviews.take(10)

[('r0pPV4-xj1sD_uGXVYxOaw', 24686),
 ('Gs4OijDfrHzAbocJ-YYGog', 21960),
 ('_-SjDVit-tv9nl4qFtltZQ', 18255),
 ('wqeLjhyY3d6ac2C8dxCBlw', 14545),
 ('LPovj-Wa7xXoJKM3qRy1bA', 14378),
 ('kbmUOqtVz4rB64zUpFKtnA', 13793),
 ('oiZUTKnsIilXwyN-HCK55w', 13715),
 ('wl7VR72-u8ADycZAwGC8gQ', 12104),
 ('u6njf5K-4W8w7p-MphTNng', 10682),
 ('AiVl1U26I_O4WqKnIZ73vw', 10535)]

In [ ]:
top_reviews_asc = user_review_counts_sort_rdd.sortBy(lambda x: x[1], ascending=True)
top_reviews_asc.take(10)

[('r4K9hqVUpbLnmzb7VWnPrg', 592),
 ('_iYBsrJUCYQWxAPRFGAPZg', 667),
 ('1trMVIHVfsaBRQpx8GW-XQ', 728),
 ('iRpe3fQw9pMJmxEMxiVjCA', 767),
 ('Hwn_c-F7rmus3ukJNp0Yug', 781),
 ('TJK6tVDQL2Tx9OkRGDVOAw', 826),
 ('MV4o5u9FfQhdKNbsat7JhQ', 884),
 ('jF6vY7rlJsVQ_XpkL0fbxQ', 1048),
 ('vedGQ3Y90omc80JARWGqOQ', 1082),
 ('oe_FHIBRrrKey1YqsvCTcQ', 1127)]

### Solution for Lab Assignment 2 Part 1
Please replace this sentence with the documentation of your code for Lab Assignment 2 Part 1.

## Part 2: Practice of PySpark SQL (4 points)
In this part, you need to use the techniques for Spark SQL  to read and process an external file that stores users' individual reviews on businesses from Tucson.

Specifically, the external file, `user_reviews.csv`, has six attributes that are listed below.
* `review_id`: a string-typed attribute indicating a review's ID
* `user_id`: a string-typed attribute indicating the reviewer's ID
* `business_id`: a string-typed attribute indicating the ID of the business that is reviewed
* `review_stars`: a float-typed attribute indicating the review's star rating
* `useful`: an integer-typed attribute indicating how many useful votes the review has received
* `review_text`: a string-typed attribute storing the review's text

Please use the Spark techniques we discussed during the class to finish the following tasks:
1. Read the csv file using Spark SQL's `read.csv` function into a Spark DataFrame; customize the schema based on the information provided above when reading the file; print the DataFrame's schema after reading the data; show the first 20 rows. (1 point)
2. For some unknown reasons, there are some reviews whose `review_id` **OR** `business_id` attribute is missing. Fill the missing values from the DataFrame with the string "missing" using the `fillna` transformation and drop the missing values from the DataFrame. Print how many rows are left after removing rows with missing values. (0.75 point)
3. Create a new column named `review_text_length` that stores the length of the review using the `withColumn` transformation. Notice that you need to use the `length` function provided by Spark SQL's functions library. See the [official document](https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.length.html) for how to use the function. (0.75 point)
4. Using the `groupby` transformation, group the DataFrame by `user_id` so as to calculate the following values for each user. Round each value to 2 decimals and choose a meaningful alias for each value. Sort the grouped dataframe by total useful votes (in descending order) and average star rating (in ascending order); show the first 20 rows (1.5 point).
    * average star rating the user gives
    * average review length the user gives
    * total useful votes the user has received

**Hint**: For task 2, you might want to filter the DataFrame so that only rows with non-missing `review_id` and `business_id` attributes are kept.

In [ ]:
# create a spark dataframe from a pandas dataframe
import pandas as pd

## first, read the external csv file into a pandas dataframe
pdf = pd.read_csv('user_reviews.csv')

## then, convert the pandas dataframe to a spark dataframe
df = spark.createDataFrame(pdf)
df.show(n=5)

## we can also convert a spark dataframe to a pandas dataframe
pdf = df.toPandas()
print(pdf.head(20))

+--------------------+--------------------+--------------------+------------+------+--------------------+
|           review_id|             user_id|         business_id|review_stars|useful|         review_text|
+--------------------+--------------------+--------------------+------------+------+--------------------+
|FTcRb7TUjE-K6spSj...|CMYCfKoEu0WF9_43z...|5Ce3lZksYVkCbrihq...|         5.0|     2|We love this litt...|
|oyxS126nYDZOL0qwP...|CMYCfKoEu0WF9_43z...|CA5BOxKRDPGJgdUQ8...|         5.0|     1|We came here for ...|
|KbFlOy2PN2dXBjdk4...|CMYCfKoEu0WF9_43z...|1MAQQhmUNU0uzHw3K...|         1.0|     4|Just a heads up t...|
|mslt0F7LpdBMQmKGk...|CMYCfKoEu0WF9_43z...|QXB4E78FXn3eotalX...|         1.0|     9|Came in to get my...|
|5SGsoqgx8CBbw6bcr...|CMYCfKoEu0WF9_43z...|M983OPfVRnwvG7zEO...|         4.0|     0|love the atmosphe...|
+--------------------+--------------------+--------------------+------------+------+--------------------+
only showing top 5 rows

                 revi

In [ ]:
from pyspark.sql.types import *

#create a spark dataframe from a csv file with inferred schema

df = spark.read.csv("user_reviews.csv", header=True, inferSchema = True)
df.show()

#print the schema (column names and types of a dataframe)

print("\n Inferred Schema of the dataframe")
df.printSchema()



+--------------------+--------------------+--------------------+------------+------+--------------------+
|           review_id|             user_id|         business_id|review_stars|useful|         review_text|
+--------------------+--------------------+--------------------+------------+------+--------------------+
|FTcRb7TUjE-K6spSj...|CMYCfKoEu0WF9_43z...|5Ce3lZksYVkCbrihq...|         5.0|     2|We love this litt...|
|oyxS126nYDZOL0qwP...|CMYCfKoEu0WF9_43z...|CA5BOxKRDPGJgdUQ8...|         5.0|     1|We came here for ...|
|KbFlOy2PN2dXBjdk4...|CMYCfKoEu0WF9_43z...|1MAQQhmUNU0uzHw3K...|         1.0|     4|Just a heads up t...|
|mslt0F7LpdBMQmKGk...|CMYCfKoEu0WF9_43z...|QXB4E78FXn3eotalX...|         1.0|     9|Came in to get my...|
|5SGsoqgx8CBbw6bcr...|CMYCfKoEu0WF9_43z...|M983OPfVRnwvG7zEO...|         4.0|     0|love the atmosphe...|
|tBfTKrhnuTB4pmjyX...|CMYCfKoEu0WF9_43z...|J-Go00lYW4f4a3lLL...|         5.0|     0|Breakfast is bomb...|
|Rmv9zoXR5ULycbak7...|aMP9YyQbzcxXTGkWE...|5Ce

In [ ]:
from pyspark.sql.types import *

In [ ]:
# create a spark dataframe from a csv file with inferred schema:
df = spark.read.csv('user_reviews.csv', header=True, inferSchema=True) # Spark will automatically detect the data type of each column.
df.show()

# print the schema (column names and types of a dataframe)
print('Inferred Schema of the DataFrame:')
df.printSchema()



+--------------------+--------------------+--------------------+------------+------+--------------------+
|           review_id|             user_id|         business_id|review_stars|useful|         review_text|
+--------------------+--------------------+--------------------+------------+------+--------------------+
|FTcRb7TUjE-K6spSj...|CMYCfKoEu0WF9_43z...|5Ce3lZksYVkCbrihq...|         5.0|     2|We love this litt...|
|oyxS126nYDZOL0qwP...|CMYCfKoEu0WF9_43z...|CA5BOxKRDPGJgdUQ8...|         5.0|     1|We came here for ...|
|KbFlOy2PN2dXBjdk4...|CMYCfKoEu0WF9_43z...|1MAQQhmUNU0uzHw3K...|         1.0|     4|Just a heads up t...|
|mslt0F7LpdBMQmKGk...|CMYCfKoEu0WF9_43z...|QXB4E78FXn3eotalX...|         1.0|     9|Came in to get my...|
|5SGsoqgx8CBbw6bcr...|CMYCfKoEu0WF9_43z...|M983OPfVRnwvG7zEO...|         4.0|     0|love the atmosphe...|
|tBfTKrhnuTB4pmjyX...|CMYCfKoEu0WF9_43z...|J-Go00lYW4f4a3lLL...|         5.0|     0|Breakfast is bomb...|
|Rmv9zoXR5ULycbak7...|aMP9YyQbzcxXTGkWE...|5Ce

In [ ]:
# create a spark dataframe from a csv file with specified schema:
schema = StructType([
    StructField('review_id', StringType(), True),
    StructField('user_id', StringType(), True),
    StructField('business_id', StringType(), True),
    StructField('review_stars', FloatType(), True),
    StructField('useful', IntegerType(), True),
    StructField('review_text',StringType(), True)
])
df = spark.read.csv('user_reviews.csv', header=True, schema=schema)
print('Specified Schema of the DataFrame:')
df.printSchema()
df.show(20)




Specified Schema of the DataFrame:
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- review_stars: float (nullable = true)
 |-- useful: integer (nullable = true)
 |-- review_text: string (nullable = true)

+--------------------+--------------------+--------------------+------------+------+--------------------+
|           review_id|             user_id|         business_id|review_stars|useful|         review_text|
+--------------------+--------------------+--------------------+------------+------+--------------------+
|FTcRb7TUjE-K6spSj...|CMYCfKoEu0WF9_43z...|5Ce3lZksYVkCbrihq...|         5.0|     2|We love this litt...|
|oyxS126nYDZOL0qwP...|CMYCfKoEu0WF9_43z...|CA5BOxKRDPGJgdUQ8...|         5.0|     1|We came here for ...|
|KbFlOy2PN2dXBjdk4...|CMYCfKoEu0WF9_43z...|1MAQQhmUNU0uzHw3K...|         1.0|     4|Just a heads up t...|
|mslt0F7LpdBMQmKGk...|CMYCfKoEu0WF9_43z...|QXB4E78FXn3eotalX...|         1.

In [ ]:
df_filled = df.fillna("missing", subset=["review_id", "business_id"])
df_filled.show(20)

+--------------------+--------------------+--------------------+------------+------+--------------------+
|           review_id|             user_id|         business_id|review_stars|useful|         review_text|
+--------------------+--------------------+--------------------+------------+------+--------------------+
|FTcRb7TUjE-K6spSj...|CMYCfKoEu0WF9_43z...|5Ce3lZksYVkCbrihq...|         5.0|     2|We love this litt...|
|oyxS126nYDZOL0qwP...|CMYCfKoEu0WF9_43z...|CA5BOxKRDPGJgdUQ8...|         5.0|     1|We came here for ...|
|KbFlOy2PN2dXBjdk4...|CMYCfKoEu0WF9_43z...|1MAQQhmUNU0uzHw3K...|         1.0|     4|Just a heads up t...|
|mslt0F7LpdBMQmKGk...|CMYCfKoEu0WF9_43z...|QXB4E78FXn3eotalX...|         1.0|     9|Came in to get my...|
|5SGsoqgx8CBbw6bcr...|CMYCfKoEu0WF9_43z...|M983OPfVRnwvG7zEO...|         4.0|     0|love the atmosphe...|
|tBfTKrhnuTB4pmjyX...|CMYCfKoEu0WF9_43z...|J-Go00lYW4f4a3lLL...|         5.0|     0|Breakfast is bomb...|
|Rmv9zoXR5ULycbak7...|aMP9YyQbzcxXTGkWE...|5Ce

In [ ]:
#number of rows
df.count()
#Drop rows from review id and business id which has missing rows

df_filled.dropna(subset=['review_id','business_id'])


1682

In [ ]:
df_filled.count()

1682

In [ ]:
from pyspark.sql.functions import length

In [ ]:
df_review_length = df_filled.withColumn("review_text_length", length(df_filled["review_text"]))

df_review_length.show(20)

+--------------------+--------------------+--------------------+------------+------+--------------------+------------------+
|           review_id|             user_id|         business_id|review_stars|useful|         review_text|review_text_length|
+--------------------+--------------------+--------------------+------------+------+--------------------+------------------+
|FTcRb7TUjE-K6spSj...|CMYCfKoEu0WF9_43z...|5Ce3lZksYVkCbrihq...|         5.0|     2|We love this litt...|               141|
|oyxS126nYDZOL0qwP...|CMYCfKoEu0WF9_43z...|CA5BOxKRDPGJgdUQ8...|         5.0|     1|We came here for ...|               488|
|KbFlOy2PN2dXBjdk4...|CMYCfKoEu0WF9_43z...|1MAQQhmUNU0uzHw3K...|         1.0|     4|Just a heads up t...|               637|
|mslt0F7LpdBMQmKGk...|CMYCfKoEu0WF9_43z...|QXB4E78FXn3eotalX...|         1.0|     9|Came in to get my...|               507|
|5SGsoqgx8CBbw6bcr...|CMYCfKoEu0WF9_43z...|M983OPfVRnwvG7zEO...|         4.0|     0|love the atmosphe...|               322|


Using the groupby transformation, group the DataFrame by user_id so as to calculate the following values for each user. Round each value to 2 decimals and choose a meaningful alias for each value. Sort the grouped dataframe by total useful votes (in descending order) and average star rating (in ascending order); show the first 20 rows (1.5 point).

    average star rating the user gives
    average review length the user gives
    total useful votes the user has received


In [ ]:
from pyspark.sql.functions import round, col
from pyspark.sql import functions as F

In [ ]:
df_grouped = df_review_length.groupBy("user_id").agg(
    round(F.avg("review_stars"), 2).alias("Average star rating"),
    round(F.avg("review_text_length"),2).alias("Average review length"),
    F.sum("useful").alias("Total useful votes"),
    F.count("*").alias("total_reviews")
)



In [ ]:
from pyspark.sql import functions as F
df_grouped_sort = df_grouped.sort('Total useful votes', 'Average review length', ascending=[False,True])
df_grouped_sort.show(20)

+--------------------+-------------------+---------------------+------------------+-------------+
|             user_id|Average star rating|Average review length|Total useful votes|total_reviews|
+--------------------+-------------------+---------------------+------------------+-------------+
|Vdy219QdKTKmzTSy1...|               4.27|               363.55|                76|           11|
|59v5-XHpKQS48vRLW...|                3.9|                619.4|                65|           10|
|oiZUTKnsIilXwyN-H...|               4.35|               806.76|                47|           17|
|IpKpHOGCqLWibbiZr...|               3.79|               528.36|                39|           14|
|lbMsbFedMjgjhWIZj...|               3.57|               652.29|                38|           14|
|cBbVbcqUWrgYLP-06...|                4.4|               1399.8|                38|            5|
|LPovj-Wa7xXoJKM3q...|               3.63|               571.63|                37|           16|
|uyVtmvKr7Hs9niun0..

### Solution for Lab Assignment 2 Part 2
Please replace this sentence with the documentation of your code for Lab Assignment 2 Part 2.

In [ ]:
"""
This code cell is for Lab Assignment 2 Part 2
"""

'\nThis code cell is for Lab Assignment 2 Part 2\n'